# 05. Prepare the wide, uncurated CT candidate-predictor universe

This notebook is step 5 of the `toronto_election_turnout` rewrite. It assembles one wide,
**uncurated** CT-level table joining:

- the Block 1-3 census profile (notebook 03's widened `ct_census_profile.csv`/`.geojson`, 622 CTs,
  restricted here to the 585-CT interpolation universe),
- the turnout/participation outcomes (notebook 04's `ct_election_outcomes.csv`; Block 4
  electoral-competitiveness has been fully retired from the pipeline this round and is not joined),
- the Block 5 (built-environment / transportation / accessibility) variables on this round's fixed
  predictor list: transportation (TTS 2022), and continuous StatCan proximity/accessibility indices.

The predictor pool this round is a **fixed, bounded list** rather than "as wide as reasonable": the
legacy KSI-collision, development-application, and 311-service-request variables, and the binary
1200m facility-access flags (library/community-centre/park/shelter), have all been retired -- they
were never part of any requested predictor family, and the park-access flag was separately confirmed
constant (=1) across all 585 CTs. Accessibility is now represented by nine continuous StatCan
proximity indices (jobs, transit, grocery, health care, childcare, elementary school, secondary
school, library, parks) plus distance to City Hall and population density, all sourced from
`census.parquet`. Transportation adds a restored no-car-household share and a new combined
walk-or-bike trip share alongside the previously ported TTS ratios.

**This notebook does no curation and no imputation.** Every raw/derived candidate column is kept,
missing values are left as `NaN`, and a missingness summary is printed as the handoff to
`06_explore_and_curate_predictors.ipynb`, which decides what actually goes into the model.

Output: `analysis/toronto_election_turnout/data/features/ct_candidate_predictors.csv` / `.geojson`
(585 rows) -- a new, wider sibling of the old `ct_model_input.csv`, not a replacement for it.


In [1]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd


In [2]:
REPO_ROOT = Path.cwd().resolve().parents[1]
DATA_ROOT = REPO_ROOT / "analysis" / "toronto_election_turnout" / "data"
ARCHIVE_ROOT = DATA_ROOT / "archive"
OUT_ROOT = REPO_ROOT / "data" / "toronto_election_turnout"  # relocated output root for this pipeline's own (non-archived) inputs/outputs

CENSUS_CSV = OUT_ROOT / "census" / "ct_census_profile.csv"
CENSUS_GEOJSON = OUT_ROOT / "census" / "ct_census_profile.geojson"
OUTCOMES_CSV = OUT_ROOT / "interpolation" / "ct_election_outcomes.csv"
TTS_ZONES_GEOJSON = REPO_ROOT / "data" / "transportation_tomorrow_survey_2022" / "tts2022zones_data.geojson"
CENSUS_PARQUET = REPO_ROOT / "data" / "census" / "census.parquet"

OUTPUT_DIR = OUT_ROOT / "features"
OUTPUT_CSV = OUTPUT_DIR / "ct_candidate_predictors.csv"
OUTPUT_GEOJSON = OUTPUT_DIR / "ct_candidate_predictors.geojson"

# Informational (non-blocking) comparison target -- see the verification section near the end.
GROUND_TRUTH_MASTER = ARCHIVE_ROOT / "variables" / "processed" / "toronto_ct_blocks_1_5_modelling_master.csv"

METRIC_CRS = 3347          # StatCan Lambert (equal-area, metres) -- used for all area/distance math


## 1. The CT universe: widened Block 1-3 census profile, restricted to the 585-CT interpolation universe

Every downstream Block 5 variable is computed only for the 585 CTs notebook 04 actually estimated
election outcomes for (`in_interpolation_universe == True`). Block 1-3 columns get the
`block1_`/`block2_`/`block3_` prefixes here (notebook 03's own output leaves them unprefixed, by
design). `ct_census_profile.csv` is considerably wider than the old pipeline's census profile --
household structure, income, labour/occupation detail, housing vintage/condition, and citizenship
detail have all been added -- so the rename dict below is a superset of the old one. Every key is
verified against the actual column list before use, so a schema mismatch between notebook 03 and this
notebook would fail loudly here rather than silently dropping a variable.

CT polygon geometry is taken directly from notebook 03's own `ct_census_profile.geojson` (EPSG:4326,
reprojected here to EPSG:3347 for area/distance work).


In [3]:
census_df = pd.read_csv(CENSUS_CSV)
census_df["ct_id"] = census_df["ct_id"].astype(float)

ct_gdf_full = gpd.read_file(CENSUS_GEOJSON)[["ct_id", "geometry"]]
ct_gdf_full["ct_id"] = ct_gdf_full["ct_id"].astype(float)

universe = census_df[census_df["in_interpolation_universe"] == True].copy().set_index("ct_id")
ct_gdf = ct_gdf_full.merge(universe.reset_index()[["ct_id"]], on="ct_id", how="inner")

ct_m = ct_gdf.to_crs(METRIC_CRS)
ct_m["geometry"] = ct_m.geometry.make_valid()  # a handful of source polygons have minor topology defects
ct_m["centroid"] = ct_m.geometry.centroid
ct_m = ct_m.set_index("ct_id")
ct_ids = list(universe.index)

print(f"CT universe: {len(ct_ids)} tracts (expect 585); all geometries valid: {ct_m.geometry.is_valid.all()}")


CT universe: 585 tracts (expect 585); all geometries valid: True


In [4]:
BLOCK123_RENAME = {
    "age_18_29_share": "block1_age_18_29_share",
    "age_30_64_share": "block1_age_30_64_share",
    "age_65_plus_share": "block1_age_65_plus_share",
    "median_age": "block1_median_age",
    "average_household_size": "block1_average_household_size",
    "bachelors_or_higher_25_64_share": "block1_bachelors_or_higher_25_64_share",
    "low_income_lim_at_share": "block1_low_income_lim_at_share",
    "unemployment_rate_share": "block1_unemployment_rate_share",
    "renter_share": "block2_renter_share",
    "owner_share": "block2_owner_share",
    "same_address_1yr_share": "block2_same_address_1yr_share",
    "same_address_5yr_share": "block2_same_address_5yr_share",
    "condo_share": "block2_condo_share",
    "apartment_share": "block2_apartment_share",
    "detached_share": "block2_detached_share",
    "semi_detached_share": "block2_semi_detached_share",
    "population_density_per_km2": "block2_population_density_per_km2",
    "structural_type_total_dwellings": "block2_structural_type_total_dwellings",
    "apartment_duplex_count": "block2_apartment_duplex_count",
    "apartment_lt5_storeys_count": "block2_apartment_lt5_storeys_count",
    "apartment_5plus_storeys_count": "block2_apartment_5plus_storeys_count",
    "apartment_total_count": "block2_apartment_total_count",
    "condo_status_total_dwellings": "block2_condo_status_total_dwellings",
    "condominium_dwellings_count": "block2_condominium_dwellings_count",
    "apartments_per_km2": "block2_apartments_per_km2",
    "condos_per_km2": "block2_condos_per_km2",
    "immigrant_share": "block3_immigrant_share",
    "recent_immigrant_share": "block3_recent_immigrant_share",
    "non_citizen_share": "block3_non_citizen_share",
    "citizen_adult_share": "block3_citizen_adult_share",
    "visible_minority_share": "block3_visible_minority_share",
    "english_french_knowledge_share": "block3_english_french_knowledge_share",
    "non_official_mother_tongue_share": "block3_non_official_mother_tongue_share",
    "subsidized_housing_tenant_share": "block5_social_housing_share",
    "transit_commute_share": "block5_census_transit_commute_share",
    "school_age_5_17_share": "block5_school_age_5_17_share",
    # NEW: widened Block 1-3 variables from notebook 03
    "married_or_common_law_share": "block1_married_or_common_law_share",
    "couple_with_children_share": "block1_couple_with_children_share",
    "one_parent_family_share": "block1_one_parent_family_share",
    "multigenerational_household_share": "block1_multigenerational_household_share",
    "one_person_household_share": "block1_one_person_household_share",
    "college_or_trades_25_64_share": "block1_college_or_trades_25_64_share",
    "labour_force_participation_rate": "block1_labour_force_participation_rate",
    "self_employment_share": "block1_self_employment_share",
    "full_time_year_round_work_share": "block1_full_time_year_round_work_share",
    "occ_management_share": "block1_occ_management_share",
    "occ_sales_service_share": "block1_occ_sales_service_share",
    "occ_trades_transport_share": "block1_occ_trades_transport_share",
    "occ_manufacturing_share": "block1_occ_manufacturing_share",
    "median_household_income": "block1_median_household_income",
    "dwelling_vintage_pre1960_share": "block2_dwelling_vintage_pre1960_share",
    "dwelling_major_repairs_needed_share": "block2_dwelling_major_repairs_needed_share",
    "core_housing_need_tenant_share": "block2_core_housing_need_tenant_share",
    "non_permanent_resident_share": "block3_non_permanent_resident_share",
    "second_generation_share": "block3_second_generation_share",
    "long_commute_60min_share": "block5_census_long_commute_60min_share",
    "work_from_home_share": "block5_census_work_from_home_share",
    # NEW this round: further widened Block 1-3 variables from notebook 03
    "never_married_share": "block1_never_married_share",
    "high_school_or_less_share": "block1_high_school_or_less_share",
    "employment_rate_share": "block1_employment_rate_share",
    "not_in_labour_force_share": "block1_not_in_labour_force_share",
    "temporary_worker_share": "block1_temporary_worker_share",
    "occ_business_finance_admin_share": "block1_occ_business_finance_admin_share",
    "occ_natural_applied_sciences_share": "block1_occ_natural_applied_sciences_share",
    "occ_education_law_social_gov_share": "block1_occ_education_law_social_gov_share",
    "occ_arts_recreation_sport_share": "block1_occ_arts_recreation_sport_share",
    "recent_mover_share": "block2_recent_mover_share",
    "high_rise_apartment_share": "block2_high_rise_apartment_share",
    "dwelling_vintage_1961_1990_share": "block2_dwelling_vintage_1961_1990_share",
    "shelter_cost_burden_30plus_share": "block2_shelter_cost_burden_30plus_share",
    "core_housing_need_share": "block2_core_housing_need_share",
    "first_generation_share": "block3_first_generation_share",
    "no_religion_share": "block3_no_religion_share",
    "non_official_language_at_home_share": "block3_non_official_language_at_home_share",
    "neither_english_nor_french_share": "block3_neither_english_nor_french_share",
}

missing_keys = [k for k in BLOCK123_RENAME if k not in census_df.columns]
print(f"BLOCK123_RENAME keys missing from ct_census_profile.csv: {missing_keys}")
assert not missing_keys, "notebook 03's census profile is missing columns this rename dict expects."

base_df = universe.rename(columns=BLOCK123_RENAME).reset_index()
base_cols = (
    ["ct_id", "ctuid", "dguid", "geo_name", "population_total", "population_18plus",
     "canadian_citizens_18plus_count", "land_area_km2"]
    + list(BLOCK123_RENAME.values())
)
base_df = base_df[base_cols].rename(columns={"canadian_citizens_18plus_count": "citizen_canadian_18plus_count"})
base_df["census_year"] = "2021"
print(f"base_df: {base_df.shape[0]} rows x {base_df.shape[1]} columns ({len(BLOCK123_RENAME)} Block 1-3/5-straggler variables)")
base_df.head(2)


BLOCK123_RENAME keys missing from ct_census_profile.csv: []
base_df: 585 rows x 84 columns (75 Block 1-3/5-straggler variables)


,ct_id,ctuid,dguid,geo_name,population_total,population_18plus,citizen_canadian_18plus_count,land_area_km2,block1_age_18_29_share,block1_age_30_64_share,...,block2_recent_mover_share,block2_high_rise_apartment_share,block2_dwelling_vintage_1961_1990_share,block2_shelter_cost_burden_30plus_share,block2_core_housing_need_share,block3_first_generation_share,block3_no_religion_share,block3_non_official_language_at_home_share,block3_neither_english_nor_french_share,census_year
0,5350128.04,5350128.04,2021S05075350128.04,128.04,4772.0,4310,3710.0,0.16,0.140461,0.528302,...,0.173459,0.938671,0.373957,0.379599,0.201754,0.459152,0.356108,0.074346,0.02199,2021
1,5350363.06,5350363.06,2021S05075350363.06,363.06,7176.0,6125,4430.0,0.82,0.188719,0.501393,...,0.178498,0.996942,0.097523,0.424765,0.233723,0.725477,0.269571,0.195122,0.05784,2021


## 2. Turnout/participation outcomes, from notebook 04

Notebook 04 now produces only the turnout/participation outcome columns, keyed by `ct_id`
(`municipal_turnout`, `provincial_turnout`, `federal_turnout`, `mean_turnout`, each election's
underlying estimated votes-cast/electors counts, and a turnout-vs-electors diagnostic ratio) -- the
Block 4 electoral-competitiveness columns (mayoral/provincial/federal margins, effective-candidate
counts, vote fragmentation) have been fully retired from the pipeline this round and are not joined
here. This is a straight left join on `ct_id`; no renaming is needed since notebook 04's column names
are already final.


In [5]:
outcomes_df = pd.read_csv(OUTCOMES_CSV)
outcomes_df["ct_id"] = outcomes_df["ct_id"].astype(float)

df = base_df.merge(outcomes_df, on="ct_id", how="left")
print(f"{df.shape[0]} rows x {df.shape[1]} columns after Block 1-3 + turnout-outcomes join")


585 rows x 98 columns after Block 1-3 + turnout-outcomes join


## 3. Block 5a: Transportation Tomorrow Survey 2022 (area-weighted zone -> CT)

`tts2022zones_data.geojson` carries every TTS metric needed as a clean, precomputed per-zone field
(`population`, `hhlds`, `veh_per_hhld`, `hhlds_no_veh` [a percentage], the `mode_*` percentages,
`trip_km_avg`, `trips_5up_per_person`) -- no need to join a separate metrics file.

Area-weighted interpolation: each zone contributes to a CT in proportion to the *share of the zone's
own area* that falls inside that CT (no population weight -- TTS zones don't publish a finer
within-zone population surface). Three different weighting schemes are used depending on what each
ratio is actually averaging:

- **household-weighted** (weight = `hhlds * portion`): no-car household share, vehicles per household
- **trip-weighted** (weight = an implied per-zone trip total, `trips_5up_per_person * population *
  portion`): transit/auto/walk-or-bike trip-mode shares
- **population-weighted** (weight = `population * portion`): average trip distance, trips per person

This round's TTS predictor list is: `no_car_household_share` (restored), `vehicles_per_household`,
`transit_trip_share`, `auto_trip_share`, a **new combined `walk_or_bike_trip_share`** (the trip-weighted
sum of `mode_walk` and `mode_bike`, replacing the walk-only share), `avg_trip_distance_km` (from
`trip_km_avg`), and `trips_per_person`. There is no TTS field for trip *duration* -- only distance --
so no duration variable is built or approximated.

Missing/null zone metrics (small-sample suppression) are filled with 0 before the weight products are
computed, so a zone with a missing field simply drops out of whichever numerator/denominator pair
depends on that field.

`.make_valid()` is still called on both zone and CT geometries for topological correctness.


In [6]:
zones_gdf = gpd.read_file(TTS_ZONES_GEOJSON)
zones_gdf["zone_id"] = zones_gdf["TTS2022"].astype(int)

RAW_ZONE_FIELDS = [
    "population", "hhlds", "veh_per_hhld", "hhlds_no_veh",
    "mode_bike", "mode_walk", "mode_transit", "mode_drive",
    "trip_km_avg", "trips_5up_per_person",
]
for col in RAW_ZONE_FIELDS:
    zones_gdf[col] = pd.to_numeric(zones_gdf[col], errors="coerce").fillna(0.0)

# Implied per-zone trip total: proxies total person-trips in the zone without a separate trips-total source.
zones_gdf["trip_weight"] = zones_gdf["trips_5up_per_person"] * zones_gdf["population"]

zones_m = zones_gdf.to_crs(METRIC_CRS)
zones_m["geometry"] = zones_m.geometry.make_valid()
zones_m["zone_area_m2"] = zones_m.geometry.area
zones_m = zones_m[zones_m["zone_area_m2"] > 0].copy()

print(f"TTS zones: {len(zones_m)} usable zones (zone_area_m2 > 0) out of {len(zones_gdf)}")


TTS zones: 5790 usable zones (zone_area_m2 > 0) out of 5790


In [7]:
ZONE_KEEP_COLS = ["zone_id", "zone_area_m2", "trip_weight"] + RAW_ZONE_FIELDS

tts_overlay = gpd.overlay(
    zones_m[ZONE_KEEP_COLS + ["geometry"]],
    ct_m.reset_index()[["ct_id", "geometry"]],
    how="intersection", keep_geom_type=False,
)
tts_overlay["inter_area_m2"] = tts_overlay.geometry.area
tts_overlay = tts_overlay[tts_overlay["inter_area_m2"] > 0].copy()
tts_overlay["portion"] = tts_overlay["inter_area_m2"] / tts_overlay["zone_area_m2"]

p = tts_overlay["portion"]
tts_overlay["w_hhlds"] = tts_overlay["hhlds"] * p
tts_overlay["w_no_car_num"] = tts_overlay["hhlds"] * (tts_overlay["hhlds_no_veh"] / 100) * p
tts_overlay["w_veh_num"] = tts_overlay["hhlds"] * tts_overlay["veh_per_hhld"] * p
tts_overlay["w_trip"] = tts_overlay["trip_weight"] * p
tts_overlay["w_transit_num"] = (tts_overlay["mode_transit"] / 100) * tts_overlay["trip_weight"] * p
tts_overlay["w_auto_num"] = (tts_overlay["mode_drive"] / 100) * tts_overlay["trip_weight"] * p
# Combined walk-or-bike share: same trip-weighting scheme as the other mode shares, summing the two
# mode percentages as a share of total trips before weighting (not summed after separate weighting).
tts_overlay["w_walkbike_num"] = (
    (tts_overlay["mode_walk"] + tts_overlay["mode_bike"]) / 100 * tts_overlay["trip_weight"] * p
)
tts_overlay["w_pop"] = tts_overlay["population"] * p
tts_overlay["w_trip_km_num"] = tts_overlay["trip_km_avg"] * tts_overlay["population"] * p
tts_overlay["w_trips_pp_num"] = tts_overlay["trips_5up_per_person"] * tts_overlay["population"] * p

WEIGHT_COLS = [
    "w_hhlds", "w_no_car_num", "w_veh_num", "w_trip", "w_transit_num", "w_auto_num",
    "w_walkbike_num", "w_pop", "w_trip_km_num", "w_trips_pp_num",
]
tts_agg = tts_overlay.groupby("ct_id")[WEIGHT_COLS].sum()
tts_agg["block5_tts_overlap_area_m2"] = tts_overlay.groupby("ct_id")["inter_area_m2"].sum()
tts_agg = tts_agg.reindex(ct_ids)

tts_out = pd.DataFrame(index=tts_agg.index)
tts_out["block5_tts_no_car_household_share"] = tts_agg["w_no_car_num"] / tts_agg["w_hhlds"]
tts_out["block5_tts_vehicles_per_household"] = tts_agg["w_veh_num"] / tts_agg["w_hhlds"]
tts_out["block5_tts_transit_trip_share"] = tts_agg["w_transit_num"] / tts_agg["w_trip"]
tts_out["block5_tts_auto_trip_share"] = tts_agg["w_auto_num"] / tts_agg["w_trip"]
tts_out["block5_tts_walk_or_bike_trip_share"] = tts_agg["w_walkbike_num"] / tts_agg["w_trip"]
tts_out["block5_tts_avg_trip_distance_km"] = tts_agg["w_trip_km_num"] / tts_agg["w_pop"]
tts_out["block5_tts_trips_per_person"] = tts_agg["w_trips_pp_num"] / tts_agg["w_pop"]
tts_out["block5_tts_overlap_area_m2"] = tts_agg["block5_tts_overlap_area_m2"]

tts_out.describe()


,block5_tts_no_car_household_share,block5_tts_vehicles_per_household,block5_tts_transit_trip_share,block5_tts_auto_trip_share,block5_tts_walk_or_bike_trip_share,block5_tts_avg_trip_distance_km,block5_tts_trips_per_person,block5_tts_overlap_area_m2
count,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,585.000000,5.850000e+02
mean,0.244381,1.078325,0.181517,0.572513,0.229965,7.291515,2.095600,1.138325e+06
std,0.170116,0.370072,0.073493,0.186643,0.158488,2.265034,0.325994,1.485518e+06
min,0.000059,0.217424,0.012143,0.117568,0.003232,2.004079,1.096626,1.345171e+04
25%,0.108477,0.822470,0.125284,0.438459,0.111953,5.810167,1.849864,4.765335e+05
50%,0.213865,1.082080,0.171107,0.611182,0.176478,7.110974,2.098813,7.904549e+05
75%,0.355261,1.353175,0.232579,0.718333,0.334413,8.383623,2.340424,1.287473e+06
max,0.804936,2.191233,0.430645,0.974237,0.704355,27.179967,3.488985,2.117637e+07


**Takeaway**: seven TTS-derived columns go forward this round -- `block5_tts_no_car_household_share`
(restored) and `block5_tts_transit_trip_share` are ported concepts (same idea as the retired notebook,
recomputed from the single-source geojson -- see the informational comparison near the end).
`block5_tts_vehicles_per_household`, `block5_tts_auto_trip_share`, `block5_tts_avg_trip_distance_km`,
and `block5_tts_trips_per_person` are carried over unchanged from the prior round. The new
`block5_tts_walk_or_bike_trip_share` replaces the old walk-only share with a combined
walk-plus-bike trip-weighted share. No archived ground truth for the new combined column --
sanity-checked below by range only.


## 4. Accessibility: continuous StatCan proximity indices from `census.parquet`

`census.parquet` carries a 2021 slice on the exact same CT geography as this pipeline (confirmed
585/585 exact match on `ct` vs. our `ctuid`, both as floats like `5350128.04`). This round replaces
the old binary 1200m facility-access flags (library/community-centre/park/shelter) with nine
continuous StatCan proximity/accessibility indices plus distance to City Hall -- the binary flags are
retired entirely (one of them, park access, was separately confirmed constant across all 585 CTs, and
none were part of any requested predictor family):

- `dist__nc` -> `block5_distance_city_hall_km`: straight-line distance from City Hall, in km (the raw
  value, not the z-scored `s_dist__nc`)
- `pmi_prox_idx_emp` -> `block5_jobs_proximity_index`
- `pmi_prox_idx_transit` -> `block5_transit_proximity_index`
- `pmi_prox_idx_grocery` -> `block5_grocery_proximity_index`
- `pmi_prox_idx_health` -> `block5_health_care_proximity_index`
- `pmi_prox_idx_childcare` -> `block5_childcare_proximity_index`
- `pmi_prox_idx_educpri` -> `block5_elementary_school_proximity_index`
- `pmi_prox_idx_educsec` -> `block5_secondary_school_proximity_index`
- `pmi_prox_idx_lib` -> `block5_library_proximity_index`
- `pmi_prox_idx_parks` -> `block5_parks_proximity_index`

These are unitless StatCan proximity/access indices. Population density (`block2_population_density_per_km2`,
already joined from notebook 03 in Section 1) is the other urban-form variable in this round's list.

**New, no ground truth to check against** -- sanity-checked by join coverage and range/`describe()`
below.


In [8]:
census_parquet_df = pd.read_parquet(CENSUS_PARQUET)
cp21 = census_parquet_df[census_parquet_df["year"] == 2021].copy()
cp21["ctuid_f"] = cp21["ct"].astype(float)

PARQUET_RENAME = {
    "dist__nc": "block5_distance_city_hall_km",
    "pmi_prox_idx_emp": "block5_jobs_proximity_index",
    "pmi_prox_idx_transit": "block5_transit_proximity_index",
    "pmi_prox_idx_grocery": "block5_grocery_proximity_index",
    "pmi_prox_idx_health": "block5_health_care_proximity_index",
    "pmi_prox_idx_childcare": "block5_childcare_proximity_index",
    "pmi_prox_idx_educpri": "block5_elementary_school_proximity_index",
    "pmi_prox_idx_educsec": "block5_secondary_school_proximity_index",
    "pmi_prox_idx_lib": "block5_library_proximity_index",
    "pmi_prox_idx_parks": "block5_parks_proximity_index",
}
cp21_small = cp21[["ctuid_f"] + list(PARQUET_RENAME.keys())].rename(columns=PARQUET_RENAME)

ctuid_lookup = base_df[["ct_id", "ctuid"]].copy()
ctuid_lookup["ctuid_f"] = ctuid_lookup["ctuid"].astype(float)
parquet_df = ctuid_lookup.merge(cp21_small, on="ctuid_f", how="left").set_index("ct_id")
parquet_df = parquet_df[list(PARQUET_RENAME.values())]

n_matched = parquet_df.notna().all(axis=1).sum()
print(f"census.parquet join: {n_matched} of {len(parquet_df)} CTs matched on all {len(PARQUET_RENAME)} urban-form columns")
parquet_df.describe()


census.parquet join: 419 of 585 CTs matched on all 10 urban-form columns


,block5_distance_city_hall_km,block5_jobs_proximity_index,block5_transit_proximity_index,block5_grocery_proximity_index,block5_health_care_proximity_index,block5_childcare_proximity_index,block5_elementary_school_proximity_index,block5_secondary_school_proximity_index,block5_library_proximity_index,block5_parks_proximity_index
count,585.000000,584.000000,582.000000,569.000000,584.000000,584.000000,581.000000,485.000000,504.000000,582.000000
mean,11.184464,0.146962,0.046027,0.097176,0.052894,0.088489,0.203827,0.101252,0.110802,0.101137
std,6.057634,0.118815,0.029583,0.081807,0.080034,0.064259,0.112917,0.090734,0.064495,0.077778
min,0.260600,0.014800,0.003800,0.007000,0.000900,0.004900,0.031000,0.033900,0.046600,0.011500
25%,6.042800,0.072000,0.025525,0.040400,0.009575,0.040425,0.120900,0.042900,0.061600,0.050625
50%,11.233300,0.096000,0.039300,0.069900,0.023250,0.069900,0.176300,0.065400,0.089550,0.076250
75%,15.721200,0.195675,0.058475,0.129500,0.055925,0.128000,0.260300,0.121800,0.142325,0.122400
max,26.808901,0.796600,0.221200,0.499100,0.568100,0.413300,0.699400,0.618600,0.443800,0.443900


## 5. Assemble the full wide candidate-predictor table -- no imputation

`block5_transit_commute_share_preferred` prefers the TTS-derived transit-trip share and falls back to
the census-only commute-mode share only when the TTS value is missing or exactly zero --
`block5_no_car_household_share` is just an alias of the TTS no-car share, kept as its own column
because it's the name downstream code is likely to reference.

**No median imputation here** -- that's notebook 06's job, after it decides which candidate columns
actually get promoted into the curated predictor set. Missing values are left as `NaN`, and a
missingness summary (column, missing count, missing %) is printed below as the handoff.


In [9]:
block5 = tts_out.join([parquet_df], how="outer")

block5["block5_transit_commute_share_preferred"] = np.where(
    block5["block5_tts_transit_trip_share"].fillna(0) != 0,
    block5["block5_tts_transit_trip_share"],
    df.set_index("ct_id")["block5_census_transit_commute_share"].reindex(block5.index),
)
block5["block5_no_car_household_share"] = block5["block5_tts_no_car_household_share"]
block5 = block5.reset_index().rename(columns={"index": "ct_id"})

full = df.merge(block5, on="ct_id", how="left")
full["block5_social_housing_note"] = (
    "Census Profile characteristic 1491: percent of tenant households in subsidized housing, "
    "converted to a 0-1 share. Denominator is tenant households, not all households."
)

print(f"Full wide candidate-predictor table (no imputation): {full.shape[0]} rows x {full.shape[1]} columns")
full.head(2)


Full wide candidate-predictor table (no imputation): 585 rows x 119 columns


,ct_id,ctuid,dguid,geo_name,population_total,population_18plus,citizen_canadian_18plus_count,land_area_km2,block1_age_18_29_share,block1_age_30_64_share,...,block5_grocery_proximity_index,block5_health_care_proximity_index,block5_childcare_proximity_index,block5_elementary_school_proximity_index,block5_secondary_school_proximity_index,block5_library_proximity_index,block5_parks_proximity_index,block5_transit_commute_share_preferred,block5_no_car_household_share,block5_social_housing_note
0,5350128.04,5350128.04,2021S05075350128.04,128.04,4772.0,4310,3710.0,0.16,0.140461,0.528302,...,0.1058,0.1563,0.1204,0.1415,0.0346,0.0604,0.2143,0.297308,0.431221,Census Profile characteristic 1491: percent of...
1,5350363.06,5350363.06,2021S05075350363.06,363.06,7176.0,6125,4430.0,0.82,0.188719,0.501393,...,0.0226,0.0160,0.0060,0.1489,0.0526,0.1044,0.0996,0.155979,0.176784,Census Profile characteristic 1491: percent of...


In [10]:
missing_summary = (
    full.isna().sum().rename("missing_count").to_frame()
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)
missing_summary["missing_pct"] = (missing_summary["missing_count"] / len(full) * 100).round(1)
print(f"{len(missing_summary)} of {full.shape[1]} columns have at least one missing value (out of {len(full)} rows)")
pd.set_option("display.max_rows", None)
missing_summary


79 of 119 columns have at least one missing value (out of 585 rows)


,missing_count,missing_pct
block5_secondary_school_proximity_index,100,17.1
block5_library_proximity_index,81,13.8
block5_grocery_proximity_index,16,2.7
block5_elementary_school_proximity_index,4,0.7
block5_parks_proximity_index,3,0.5
block5_census_long_commute_60min_share,3,0.5
block5_census_transit_commute_share,3,0.5
block5_transit_proximity_index,3,0.5
citizen_canadian_18plus_count,2,0.3
block1_average_household_size,2,0.3


## 6. Save outputs

Written as a **new** file, `ct_candidate_predictors.csv`/`.geojson` -- deliberately not overwriting
`ct_model_input.csv`, which stays notebook 06's job to produce from this wider, uncurated table.


In [11]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
full.to_csv(OUTPUT_CSV, index=False)
print(f"Wrote {full.shape[0]} rows x {full.shape[1]} columns to {OUTPUT_CSV.relative_to(REPO_ROOT)}")

geometry_lookup = ct_gdf.set_index("ct_id").geometry  # EPSG:4326, from notebook 03
full_gdf = gpd.GeoDataFrame(
    full, geometry=[geometry_lookup[cid] for cid in full["ct_id"]], crs=4326
)
full_gdf.to_file(OUTPUT_GEOJSON, driver="GeoJSON")
print(f"Wrote geometry-carrying copy to {OUTPUT_GEOJSON.relative_to(REPO_ROOT)}")


Wrote 585 rows x 119 columns to data/toronto_election_turnout/features/ct_candidate_predictors.csv


Wrote geometry-carrying copy to data/toronto_election_turnout/features/ct_candidate_predictors.geojson


## 7. Verification

Two informational (non-blocking) checks -- not hard pass/fail gates, since the TTS computation now
draws on a single source file with a different internal weighting scheme than the archived pipeline,
so small-to-moderate differences from the archive are expected and fine for those columns. Block 4
(competitiveness), facility-access flags, KSI/development-applications, and 311 are no longer part of
this notebook, so there is nothing to check for them here.

1. **Comparison against the archived `toronto_ct_blocks_1_5_modelling_master.csv`** for every column
   that has a directly comparable archived counterpart -- reported as max-abs-diff, informational only.
2. **Range/coverage checks for genuinely new/changed columns** (the restored no-car-household share,
   vehicles/household, auto trip share, the new combined walk-or-bike trip share, average trip
   distance, trips per person; the nine proximity indices; distance to city hall) -- there's nothing
   to verify most of these against, so this just confirms sane ranges and full 585-row coverage.


In [12]:
gt_master = pd.read_csv(GROUND_TRUTH_MASTER)
gt_master["ct_id"] = gt_master["ct_id"].astype(float)
gt_master_idx = gt_master.set_index("ct_id")
full_idx = full.set_index("ct_id")

NOTE_COLUMNS = {"block5_social_housing_note"}
comparable_cols = [
    c for c in full.columns
    if c.startswith("block5_") and c in gt_master.columns and c not in NOTE_COLUMNS
]

rows = []
for col in comparable_cols:
    a = pd.to_numeric(full_idx[col], errors="coerce")
    b = pd.to_numeric(gt_master_idx[col], errors="coerce")
    common = a.index.intersection(b.index)
    diff = (a.loc[common] - b.loc[common]).abs()
    rows.append({"variable": col, "max_abs_diff": diff.max(), "n_compared": int(diff.notna().sum())})

comparison_report = pd.DataFrame(rows).sort_values("max_abs_diff", ascending=False).reset_index(drop=True)
print("Informational comparison against archived toronto_ct_blocks_1_5_modelling_master.csv")
print("(no hard pass/fail gate here -- see section 7 markdown above for why)\n")
print(comparison_report.to_string(index=False))


Informational comparison against archived toronto_ct_blocks_1_5_modelling_master.csv
(no hard pass/fail gate here -- see section 7 markdown above for why)

                              variable  max_abs_diff  n_compared
            block5_tts_overlap_area_m2  2.194927e+06         585
     block5_tts_no_car_household_share  2.041431e-01         585
         block5_no_car_household_share  2.041431e-01         585
         block5_tts_transit_trip_share  1.273799e-01         585
block5_transit_commute_share_preferred  1.273799e-01         585
          block5_school_age_5_17_share  4.996550e-11         585
   block5_census_transit_commute_share  4.985162e-11         582
           block5_social_housing_share  1.110223e-16         583


In [13]:
ALL_TTS_COLS = [
    "block5_tts_no_car_household_share",
    "block5_tts_vehicles_per_household",
    "block5_tts_transit_trip_share",
    "block5_tts_auto_trip_share",
    "block5_tts_walk_or_bike_trip_share",
    "block5_tts_avg_trip_distance_km",
    "block5_tts_trips_per_person",
]
NEW_PARQUET_COLS = list(PARQUET_RENAME.values())

print(f"TTS-derived variables (all {len(ALL_TTS_COLS)}) -- range/coverage check:")
print(full[ALL_TTS_COLS].describe())
print()
print("census.parquet urban-form/accessibility variables -- no archived ground truth, range/coverage check only:")
print(full[NEW_PARQUET_COLS].describe())


TTS-derived variables (all 7) -- range/coverage check:
       block5_tts_no_car_household_share  block5_tts_vehicles_per_household  \
count                         585.000000                         585.000000   
mean                            0.244381                           1.078325   
std                             0.170116                           0.370072   
min                             0.000059                           0.217424   
25%                             0.108477                           0.822470   
50%                             0.213865                           1.082080   
75%                             0.355261                           1.353175   
max                             0.804936                           2.191233   

       block5_tts_transit_trip_share  block5_tts_auto_trip_share  \
count                     585.000000                  585.000000   
mean                        0.181517                    0.572513   
std                         0

## Takeaways for the next notebook

- This notebook is deliberately **wide and uncurated**, but the predictor pool is now a **fixed,
  bounded list** rather than "as wide as reasonable": every candidate column on this round's
  requested-variable list is included, with no feature selection and no imputation.
- **Block 4** (electoral competitiveness: margins, effective-candidate counts, fragmentation) is fully
  retired from the pipeline -- notebook 04 no longer produces it, and this notebook no longer joins it.
- **KSI collisions, development applications, and 311 service requests** are fully removed -- these
  were exploratory/completeness-only variables never part of any requested predictor family.
- **Binary facility-access flags** (library/community-centre/park/shelter within 1200m) are fully
  removed, replaced by nine continuous StatCan proximity indices from `census.parquet` (jobs, transit,
  grocery, health care, childcare, elementary school, secondary school, library, parks) plus distance
  to City Hall -- the park-access flag in particular had been confirmed constant (=1) across all 585
  CTs in a prior round.
- Transportation (TTS) now outputs a restored `block5_tts_no_car_household_share` and a new combined
  `block5_tts_walk_or_bike_trip_share` (replacing a walk-only share), alongside the previously ported
  vehicles/household, transit/auto trip shares, average trip distance, and trips per person. There is
  no TTS trip-duration field, so no duration variable is built.
- Ported TTS columns (no-car/transit shares) were checked informationally against the archived
  `toronto_ct_blocks_1_5_modelling_master.csv` -- see Section 7 for the max-abs-diff table. Small
  differences are expected given the TTS computation's single-source-file simplification and are not
  treated as failures.
- Genuinely new/changed columns (the combined walk-or-bike share, the nine `census.parquet` proximity
  indices, distance to city hall) have no archived ground truth and were sanity-checked by range and
  585-row coverage only.
- Output: `analysis/toronto_election_turnout/data/features/ct_candidate_predictors.csv` / `.geojson`,
  585 rows -- the wide input `06_explore_and_curate_predictors.ipynb` will read, curate, and (where
  appropriate) impute.
